# Hands-on: Antenna Placement
## Constrain total number of active antennas to be $\le F$

Modify the tutorial notebook and include the constraint as explained in the slides.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import math
import random
from collections import Counter

from pyqubo import Array
import neal

## Problem formulation

### Generate Antenna Layout

In [ ]:
SEED = 42 # you might change the SEED to generate different layouts

np.random.seed(SEED)
random.seed(SEED)

# Number of candidate antennas
N = 20

# Create graph (positions only for visualization)
GR = nx.Graph()
GR.add_nodes_from(range(N))

# Random 2D positions
pos = nx.random_layout(GR, seed=SEED)

# Assign random signal radii
R = np.random.uniform(0.1, 0.3, size=N)

# Coverage area
Area = math.pi * R**2

Visualize All Candidate Antennas

In [ ]:
plt.figure(figsize=(6,6))
nx.draw(GR, pos=pos, with_labels=True)

ax = plt.gca()
for i in range(N):
    circle = plt.Circle(pos[i], R[i], alpha=0.3)
    ax.add_patch(circle)

plt.axis('equal')
plt.title("All Candidate Antennas")
plt.show()

Compute Interference Matrix:
- Two antennas interfere if $R_i + R_j > d(i,j)$ with $d(i,j)$ the distance between the two positions

In [ ]:
def distance(pos, i, j):
    return np.linalg.norm(pos[i] - pos[j])

Rho = np.zeros((N, N))
Noise = np.zeros((N, N))

for i in range(N):
    for j in range(i+1, N):
        overlap = R[i] + R[j] - distance(pos, i, j)
        if overlap > 0:
            Rho[i, j] = overlap
            Rho[j, i] = overlap
            Noise[i, j] = math.pi * overlap**2
            Noise[j, i] = Noise[i, j]

Interference Graph: if there is a connection between two nodes it means there is interference between the two corresponding antennas

In [ ]:
GRe = nx.Graph()
GRe.add_nodes_from(range(N))

for i in range(N):
    for j in range(i+1, N):
        if Noise[i, j] > 0:
            GRe.add_edge(i, j)

plt.figure(figsize=(6,6))
nx.draw(GRe, pos=pos, with_labels=True)
plt.title("Interference Graph")
plt.show()

### Build the QUBO Model

In [ ]:
alpha = 1  # trade-off parameter for interference

# Fix maximum number of active antennas
F = 15
beta = 0.5 # penalty constant - you might change it to see the effect

# We need F additional "ghost" qubits
Ntot = N + F

# Define Ntot Binary variables
q = Array.create('q', shape=Ntot, vartype='BINARY')

# Constraining some antennas to be on
L= np.zeros(N)
# Selecting antennas 0,12,8 (you might change selection, add or remove antennas)
L[0] = 1
L[12] = 1
L[8] = 1
gamma = 1  # Penalty constant - you might change it to see the effect

# MODIFY THE HAMILTONIAN H BY INCLUDING PENALTY TERMS THAT SELECT SOLUTIONS
# WITH AT MOST F ANTENNAS
H = 0

# Linear term
for i in range(N):
    H += -Area[i] * q[i]
    H -= gamma * L[i] * q[i]
for i in range(Ntot):
    # WRITE YOUR CODE HERE

# Quadratic term
for i in range(N):
    for j in range(i+1, N):
        # interference penalty
        if Noise[i,j] > 0:
            H += alpha * Noise[i,j] * q[i] * q[j]
for i in range(Ntot):
    for j in range(i+1, Ntot):
        # WRITE YOUR CODE HERE

Compile

In [ ]:
model = H.compile()
bqm = model.to_bqm() #traslate the model to bqm

## Sampling
### Simulated Annealing

In [ ]:
sa = neal.SimulatedAnnealingSampler()
sampleset = sa.sample(bqm, num_reads=5000)

decoded = model.decode_sampleset(sampleset)
best = min(decoded, key=lambda x: x.energy)

sol_sa = np.array([best.sample[f'q[{i}]'] for i in range(N)])

print("Best solution:", sol_sa)
print("Energy:", best.energy)
if beta > 0:
    print(f"Number of selected antennas is {np.sum(sol_sa)}, should be less or equal to F={F}")
for i in range(0,N):
    if L[i] == 1:
        if sol_sa[i] == 1:
            print(f"Antenna {i} correctly turned on")
        else:
            print(f"Antenna {i} NOT turned on - CONSTRAINT NOT SATISFIED")

Visualize optimal selection

In [ ]:
plt.figure(figsize=(6,6))
nx.draw(GR, pos=pos, with_labels=True)

ax = plt.gca()
for i in range(N):
    if sol_sa[i] == 1:
        circle = plt.Circle(pos[i], R[i], alpha=0.4)
        ax.add_patch(circle)

plt.axis('equal')
plt.title("Selected Antennas (SA)")
plt.show()

Analyze Coverage vs Interference

In [ ]:
total_area = np.sum(Area * sol_sa)

total_interference = 0
for i in range(N):
    for j in range(i+1, N):
        total_interference += Noise[i, j] * sol_sa[i] * sol_sa[j]

print("Total coverage area:", total_area)
print("Total interference:", total_interference)

### Quantum Annealing on D-Wave QPU

In [ ]:
from dwave.system import DWaveSampler, EmbeddingComposite

# Connect to QPU
qpu = DWaveSampler()

# Wrap with embedding composite
sampler = EmbeddingComposite(qpu)

# Submit problem
sampleset_qpu = sampler.sample(
    bqm,
    num_reads=1000,
    annealing_time=20,        # in μs
    return_embedding=True
)

In [ ]:
# Analize solution
decoded_qpu = model.decode_sampleset(sampleset_qpu)
best_qpu = min(decoded_qpu, key=lambda x: x.energy)

sol_qpu = np.array([best_qpu.sample[f'q[{i}]'] for i in range(N)])

print("QPU solution:", sol_qpu)
print("Energy:", best_qpu.energy)
if beta > 0:
    print(f"Number of selected antennas is {np.sum(sol_sa)}, should be less or equal to F={F}")
for i in range(0,N):
    if L[i] == 1:
        if sol_qpu[i] == 1:
            print(f"Antenna {i} correctly turned on")
        else:
            print(f"Antenna {i} NOT turned on - CONSTRAINT NOT SATISFIED")

In [ ]:
if np.array_equal(sol_qpu, sol_sa):
    print("SA and QPU solutions are EQUAL!")
else:
    print("SA and QPU solutions are DIFFERENT!")
    plt.figure(figsize=(6,6))
    nx.draw(GR, pos=pos, with_labels=True)
    
    ax = plt.gca()
    for i in range(N):
        if sol_qpu[i] == 1:
            circle = plt.Circle(pos[i], R[i], alpha=0.4)
            ax.add_patch(circle)
    
    plt.axis('equal')
    plt.title("Selected Antennas (QPU)")
    plt.show()

#### Inspect embedding

In [ ]:
embedding = sampleset_qpu.info["embedding_context"]["embedding"]

print("Number of logical variables:", len(embedding))

# Compute physical qubits used
physical_qubits_used = sum(len(chain) for chain in embedding.values())

print("Total physical qubits used:", physical_qubits_used)
print("Average chain length:", physical_qubits_used / len(embedding))

In [ ]:
chain_lengths = [len(chain) for chain in embedding.values()]

plt.figure()
plt.hist(chain_lengths, bins=range(1, max(chain_lengths)+2))
plt.xlabel("Chain length")
plt.ylabel("Number of logical variables")
plt.title("Distribution of Chain Lengths")
plt.show()

In [ ]:
# Fraction of broken chains per sample
chain_breaks = sampleset_qpu.record.chain_break_fraction

print("Average chain break fraction:", np.mean(chain_breaks))
print("Max chain break fraction:", np.max(chain_breaks))

In [ ]:
plt.figure()
plt.hist(chain_breaks, bins=20)
plt.xlabel("Chain break fraction")
plt.ylabel("Counts")
plt.title("Chain Break Distribution")
plt.show()

In [ ]:
# Extract raw energies
raw_energies = sampleset_qpu.record.energy

# Find index of best energy
best_index = np.argmin(raw_energies)

print("Best raw energy:", raw_energies[best_index])

best_chain_break = sampleset_qpu.record.chain_break_fraction[best_index]

print("Chain break fraction of best sample:", best_chain_break)